In [1]:
"""
Figure 5.1 — Four-panel model performance overview (v6)
"""
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

DATA = "Github"
pred = pd.read_csv(f"{DATA}/predictions.csv", parse_dates=["date"])
fi   = pd.read_csv(f"{DATA}/feature_importance.csv")
wm   = pd.read_csv(f"{DATA}/warning_metrics.csv")

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 9,
    "axes.titlesize": 10, "axes.titleweight": "bold", "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.12, "grid.linewidth": 0.4,
    "legend.fontsize": 7.5, "legend.framealpha": 0.92, "legend.edgecolor": "#cccccc",
    "figure.dpi": 300,
})

C_ACTUAL="#2c2c2c"; C_CAL_RF="#1a6faf"; C_RF_RAW="#7bafd4"
C_GARCH="#c44e52"; C_THRESH="#d4a02e"; C_ANNO="#b03030"

fig = plt.figure(figsize=(11.5, 7.2))
gs = gridspec.GridSpec(2, 4, hspace=0.40, wspace=0.35,
                       left=0.06, right=0.97, top=0.94, bottom=0.07)
ax_a  = fig.add_subplot(gs[0, 0:2])
ax_b1 = fig.add_subplot(gs[0, 2])
ax_b2 = fig.add_subplot(gs[0, 3], sharey=ax_b1)
ax_c  = fig.add_subplot(gs[1, 0:2])
ax_d  = fig.add_subplot(gs[1, 2:4])

# ══ (A) Time series + GARCH-lag annotation ══════════════════════
ax_a.plot(pred["date"], pred["spy_actual_vol"], color=C_ACTUAL,
          linewidth=0.5, alpha=0.35, label="Actual")
ax_a.plot(pred["date"], pred["spy_calibrated_rf_pred"], color=C_CAL_RF,
          linewidth=0.7, label="Calibrated RF")
ax_a.plot(pred["date"], pred["spy_garch_pred"], color=C_GARCH,
          linewidth=0.6, alpha=0.50, label="GARCH(1,1)")
ax_a.axhline(0.15, color=C_THRESH, ls="--", lw=0.7, alpha=0.7, label="Warning (15%)")
ax_a.set_ylabel("Annualised volatility")
ax_a.set_title("(A)  SPY: actual vs. predicted volatility")
ax_a.legend(loc="upper left", frameon=True)
ax_a.xaxis.set_major_locator(mdates.YearLocator())
ax_a.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax_a.set_ylim(0, 0.85)

# Annotation: post-2025-spike region, GARCH still high, Cal RF tracked down
anno_date = pd.Timestamp("2025-04-25")
anno_x = mdates.date2num(anno_date)

# Arrow to GARCH (still high ~0.40)
ax_a.annotate(
    "GARCH slow to revert",
    xy=(anno_x, 0.40), xytext=(mdates.date2num(pd.Timestamp("2024-01-15")), 0.72),
    fontsize=7, color=C_GARCH,
    arrowprops=dict(arrowstyle="-|>", color=C_GARCH, lw=0.8,
                    connectionstyle="arc3,rad=-0.15"),
    bbox=dict(fc="white", ec=C_GARCH, alpha=0.88, pad=2.5, lw=0.5,
              boxstyle="round,pad=0.3"),
)

# Arrow to Cal RF (tracked down ~0.19)
ax_a.annotate(
    "Cal. RF tracks decline",
    xy=(anno_x, 0.19), xytext=(mdates.date2num(pd.Timestamp("2024-01-15")), 0.58),
    fontsize=7, color=C_CAL_RF,
    arrowprops=dict(arrowstyle="-|>", color=C_CAL_RF, lw=0.8,
                    connectionstyle="arc3,rad=-0.15"),
    bbox=dict(fc="white", ec=C_CAL_RF, alpha=0.88, pad=2.5, lw=0.5,
              boxstyle="round,pad=0.3"),
)

# ══ (B1/B2) Scatter ═════════════════════════════════════════════
scatter_kw = dict(s=5, alpha=0.25, edgecolors="none", zorder=2)
diag_kw    = dict(ls="--", lw=0.6, color="#999999", zorder=1)
lim = 0.88
rf_min, rf_max = pred["spy_rf_pred"].min(), pred["spy_rf_pred"].max()
cal_min, cal_max = pred["spy_calibrated_rf_pred"].min(), pred["spy_calibrated_rf_pred"].max()

ax_b1.scatter(pred["spy_rf_pred"], pred["spy_actual_vol"], color=C_RF_RAW, **scatter_kw)
ax_b1.plot([0, lim], [0, lim], **diag_kw)
ax_b1.set_xlabel("Predicted vol"); ax_b1.set_ylabel("Actual vol")
ax_b1.set_title("(B1)  Raw RF", fontsize=9.5)
ax_b1.set_xlim(0, lim); ax_b1.set_ylim(0, lim)
ax_b1.annotate("", xy=(rf_max, 0.04), xytext=(rf_min, 0.04),
               arrowprops=dict(arrowstyle="<->", color=C_RF_RAW, lw=1.3))
ax_b1.text((rf_min+rf_max)/2, 0.07, f"{rf_min:.2f} – {rf_max:.2f}",
           ha="center", fontsize=6.5, color="#555555",
           bbox=dict(fc="white", ec="none", alpha=0.7, pad=1))

ax_b2.scatter(pred["spy_calibrated_rf_pred"], pred["spy_actual_vol"], color=C_CAL_RF, **scatter_kw)
ax_b2.plot([0, lim], [0, lim], **diag_kw)
ax_b2.set_xlabel("Predicted vol")
ax_b2.set_title("(B2)  Calibrated RF", fontsize=9.5)
ax_b2.set_xlim(0, lim)
plt.setp(ax_b2.get_yticklabels(), visible=False)
ax_b2.annotate("", xy=(cal_max, 0.04), xytext=(cal_min, 0.04),
               arrowprops=dict(arrowstyle="<->", color=C_CAL_RF, lw=1.3))
ax_b2.text((cal_min+cal_max)/2, 0.07, f"{cal_min:.2f} – {cal_max:.2f}",
           ha="center", fontsize=6.5, color="#555555",
           bbox=dict(fc="white", ec="none", alpha=0.7, pad=1))

# ══ (C) Feature importance ══════════════════════════════════════
spy_fi = fi[fi["asset"]=="SPY"].head(8)
tlt_fi = fi[fi["asset"]=="TLT"].head(8)
top_features = list(dict.fromkeys(list(spy_fi["feature"])+list(tlt_fi["feature"])))[:10]
spy_vals, tlt_vals = [], []
for f in top_features:
    sv = fi[(fi["asset"]=="SPY")&(fi["feature"]==f)]["importance"]
    tv = fi[(fi["asset"]=="TLT")&(fi["feature"]==f)]["importance"]
    spy_vals.append(sv.values[0]*100 if len(sv) else 0)
    tlt_vals.append(tv.values[0]*100 if len(tv) else 0)

y_pos = np.arange(len(top_features)); bar_h = 0.34
bars_spy = ax_c.barh(y_pos+bar_h/2, spy_vals, bar_h, color=C_CAL_RF, alpha=0.82,
                      edgecolor="#ffffff", linewidth=0.4, label="SPY")
bars_tlt = ax_c.barh(y_pos-bar_h/2, tlt_vals, bar_h, color=C_GARCH, alpha=0.50,
                      edgecolor="#ffffff", linewidth=0.4, label="TLT")
for bar, v in zip(bars_spy, spy_vals):
    if v > 1.5:
        ax_c.text(bar.get_width()+0.4, bar.get_y()+bar.get_height()/2,
                  f"{v:.1f}", va="center", fontsize=6.5, color="#444444")
for bar, v in zip(bars_tlt, tlt_vals):
    if v > 1.5:
        ax_c.text(bar.get_width()+0.4, bar.get_y()+bar.get_height()/2,
                  f"{v:.1f}", va="center", fontsize=6.5, color="#444444")
ax_c.set_yticks(y_pos)
ax_c.set_yticklabels([f.replace("_"," ") for f in top_features], fontsize=8)
ax_c.invert_yaxis(); ax_c.set_xlabel("Importance (%)")
ax_c.set_title("(C)  Feature importance comparison")
ax_c.legend(loc="lower right", frameon=True); ax_c.set_xlim(0, 40)

# ══ (D) Warning P/R/F1 ══════════════════════════════════════════
models_show = ["Calibrated RF","Random Forest","GARCH(1,1)","Baseline"]
spy_wm = wm[(wm["asset"]=="SPY")&(wm["model"].isin(models_show))]
spy_wm = spy_wm.set_index("model").loc[models_show].reset_index()
x = np.arange(len(models_show)); w = 0.23
colors_prf = ["#4e95cc","#e8874a","#6aab47"]
edge_prf   = ["#3a7aad","#c96e35","#508c30"]
for i, (metric, label) in enumerate(zip(["precision","recall","f1"],
                                         ["Precision","Recall","F1"])):
    vals = spy_wm[metric].values
    bars = ax_d.bar(x+(i-1)*w, vals, w, label=label,
                    color=colors_prf[i], alpha=0.85, edgecolor=edge_prf[i], linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax_d.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                  f"{v:.2f}", ha="center", va="bottom", fontsize=6, color="#444444")
ax_d.set_xticks(x)
ax_d.set_xticklabels(["Cal. RF","RF","GARCH","Baseline"], fontsize=8.5)
ax_d.set_ylabel("Score"); ax_d.set_ylim(0.55, 0.83)
ax_d.set_title("(D)  SPY early warning")
ax_d.legend(loc="lower right", frameon=True, ncol=3)

out = "fig5_v6.png"
fig.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Saved: {out}"); plt.close()

Saved: fig5_v6.png
